# ANN-SNN Scaling Laws — Run Everything

**Just click Runtime > Run all. Wait ~20 min. Download zip at the end.**

Make sure: Runtime > Change runtime type > **T4 GPU**

In [ ]:
# Setup
!rm -rf /content/ann_snn_scaling_laws
!git clone https://github.com/Vighneshwarkuru/ann_snn_scaling_laws.git /content/ann_snn_scaling_laws
%cd /content/ann_snn_scaling_laws
!pip install -q spikingjelly scipy pandas matplotlib seaborn pyyaml ptflops tqdm psutil
import torch; print(f"\nGPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "\n!! NO GPU — Go to Runtime > Change runtime type > T4 GPU !!")

In [ ]:
# Run all experiments (~15 min on T4)
!rm -rf results_real/raw/* checkpoints/*
!python scripts/run_real_experiment.py --quick

In [ ]:
# Show results
import pandas as pd, glob, os
from pathlib import Path

csvs = sorted(glob.glob('results_real/raw/*.csv'))
print(f"Completed: {len(csvs)} experiments\n")

df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
pivot = df.pivot_table(values='snn_accuracy', index=['model','dataset'], columns='T', aggfunc='mean')
print((pivot * 100).round(2).to_string())

In [ ]:
# Generate plots
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, dataset in enumerate(sorted(df['dataset'].unique())):
    ax = axes[i]
    ax.set_title(dataset.upper())
    for model in sorted(df['model'].unique()):
        sub = df[(df['dataset']==dataset) & (df['model']==model)].sort_values('T')
        ax.plot(sub['T'], sub['snn_accuracy']*100, 'o-', label=model, markersize=6, linewidth=2)
    ann_acc = df[df['dataset']==dataset]['ann_accuracy'].mean()*100
    ax.axhline(y=ann_acc, color='red', linestyle='--', alpha=0.5, label=f'ANN {ann_acc:.1f}%')
    ax.set_xlabel('Timesteps (T)'); ax.set_ylabel('Accuracy (%)')
    ax.set_xscale('log', base=2); ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 105)
plt.suptitle('Scaling Laws: Accuracy vs Timesteps (REAL DATA)', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.savefig('paper/figures/real_scaling_curves.png', dpi=150); plt.show()

In [ ]:
# Download everything
!zip -qr /content/results.zip results_real/ paper/figures/ checkpoints/
from google.colab import files
files.download('/content/results.zip')
print("\nDone! Unzip into your project folder.")